In [ ]:
# Import required python libraries
import pandas as pd
import numpy as np

In [ ]:
# Load Datasets
customers = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Banking Customer Insights/Datasets/customers.csv')
transactions = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Banking Customer Insights/Datasets/transactions.csv')
churn = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Banking Customer Insights/Datasets/churn_data.csv')

In [ ]:
# Basic Data Inspection
print('\n--- Customers ---')
print(customers.head(3))

print('\n--- Transactions ---')
print(transactions.head(3))

print('\n--- Churn ---')
print(churn.head(3))


--- Customers ---
   CustomerID              Name  Gender  Age           City Region  Tenure  \
0     7452884   Crystal Bernard  Female   68        Kolkata   East       2   
1     2937289  Jennifer Daniels  Female   53          Delhi  North       5   
2     1891376    Michael Morgan    Male   24  Visakhapatnam   East       9   

   Balance  CreditScore IsActive  Salary           Phone  
0   285698          610       No  278348  +91 9482823064  
1   250894          497      Yes  733564  +91 9559366219  
2   137945          498      Yes  619623  +91 9427324351  

--- Transactions ---
   TransactionID  CustomerID        Date TransactionType  Amount Channel
0         739888     6207880  2025-06-22          Credit    1043  Branch
1         931921     2747685  2024-12-22           Debit    5173  Branch
2         847226     9828815  2025-05-27          Credit    2013  Branch

--- Churn ---
   CustomerID LastTransactionDate  LastTransactionDays  Complaints  Products  \
0     7452884          

In [ ]:
# Check for missing values
print('\nMissing Values Summary')
print('\nCustomers\n',customers.isnull().sum())
print('\nTransactions\n',transactions.isnull().sum())
print('\nChurn\n',churn.isnull().sum())


Missing Values Summary

Customers
 CustomerID     0
Name           0
Gender         0
Age            0
City           0
Region         0
Tenure         0
Balance        0
CreditScore    0
IsActive       0
Salary         0
Phone          0
dtype: int64

Transactions
 TransactionID      0
CustomerID         0
Date               0
TransactionType    0
Amount             0
Channel            0
dtype: int64

Churn
 CustomerID             0
LastTransactionDate    0
LastTransactionDays    0
Complaints             0
Products               0
Exited                 0
dtype: int64


In [ ]:
# Merge Datasets

# Merge churn data with customers (1 to 1)
df = pd.merge(customers,churn,on='CustomerID',how='left')

# Merge with transactions (many to one relationship)
# Calculate total transactions and total spend per customer
txn_summary = transactions.groupby('CustomerID').agg(
    TotalTransactions = ('TransactionID','count'),
    TotalTransactionAmount = ('Amount','sum'),
    AvgTransactionAmount = ('Amount','mean')
).reset_index()

# Merge transaction summary into master dataset
df = pd.merge(df,txn_summary,on='CustomerID',how='left')

print('Final shape:{df.shape}')
df.head(5)

Final shape:{df.shape}


,CustomerID,Name,Gender,Age,City,Region,Tenure,Balance,CreditScore,IsActive,Salary,Phone,LastTransactionDate,LastTransactionDays,Complaints,Products,Exited,TotalTransactions,TotalTransactionAmount,AvgTransactionAmount
0,7452884,Crystal Bernard,Female,68,Kolkata,East,2,285698,610,No,278348,+91 9482823064,1970-01-01 00:00:00.000000001,1,0,4,0,23,107702,4682.695652
1,2937289,Jennifer Daniels,Female,53,Delhi,North,5,250894,497,Yes,733564,+91 9559366219,1970-01-01 00:00:00.000000011,11,0,2,0,15,73240,4882.666667
2,1891376,Michael Morgan,Male,24,Visakhapatnam,East,9,137945,498,Yes,619623,+91 9427324351,1970-01-01 00:00:00.000000009,9,2,2,0,18,85506,4750.333333
3,6985516,Leslie Rowe,Female,45,Bhopal,East,8,215293,642,Yes,550371,+91 9987189900,1970-01-01 00:00:00.000000015,15,4,4,0,29,136664,4712.551724
4,4393253,Toni Taylor,Female,57,Delhi,North,7,88012,413,Yes,366789,+91 9286333961,1970-01-01 00:00:00.000000002,2,1,3,0,20,95698,4784.900000


In [ ]:
# Data Cleaning and Feature Formatting

# Convert data columns
churn['LastTransactionDate'] = pd.to_datetime(churn['LastTransactionDays'],errors='coerce')

# Fill any missing transacion summaries
df.loc[:,'TotalTransactions']=df['TotalTransactions'].fillna(0)
df.loc[:,'TotalTransactionAmount']=df['TotalTransactionAmount'].fillna(0)
df.loc[:,'AvgTransactionAmount']=df['AvgTransactionAmount'].fillna(0)

# Strandardize column names for modelling
df.rename(columns={
    'IsActive':'ActiveStatus',
    'Exited':'Churned'
},inplace=True)

# Convert categorical columns
df['Gender'] = df['Gender'].astype('category')
df['City'] = df['City'].astype('category')
df['Region'] = df['Region'].astype('category')
df['ActiveStatus'] = df['ActiveStatus'].astype('category')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   CustomerID              10000 non-null  int64         
 1   Name                    10000 non-null  object        
 2   Gender                  10000 non-null  category      
 3   Age                     10000 non-null  int64         
 4   City                    10000 non-null  category      
 5   Region                  10000 non-null  category      
 6   Tenure                  10000 non-null  int64         
 7   Balance                 10000 non-null  int64         
 8   CreditScore             10000 non-null  int64         
 9   ActiveStatus            10000 non-null  category      
 10  Salary                  10000 non-null  int64         
 11  Phone                   10000 non-null  object        
 12  LastTransactionDate     10000 non-null  datetim

In [ ]:
# Save the Final Data
df.to_csv('/content/drive/MyDrive/Colab Notebooks/Banking Customer Insights/Datasets/bank_customer_master.csv',index=False)